In [1]:
pip install pillow

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import random
import csv
import numpy as np
from PIL import Image
import math

In [2]:
# ── Configuración ─────────────────────────────────────────────────────────────
UPLOAD_DIR  = "."          # Carpeta con los 3 PNGs fuente (ajusta si es necesario)
OUTPUT_DIR  = "./registro_dataset_5_8_26_v5"
CSV_PATH    = "./registro_dataset_v5.csv"

SRC = {
    "cyan":    os.path.join(UPLOAD_DIR, "registro_cyan_4k.png"),
    "magenta": os.path.join(UPLOAD_DIR, "registro_magenta_4k.png"),
    "yellow":  os.path.join(UPLOAD_DIR, "registro_amarillo_4k.png"),
}

OFFSET_MIN  = 2    # desplazamiento mínimo en píxeles (al menos un eje)
OFFSET_MAX  = 10   # desplazamiento máximo en píxeles
SPRITE_SCALE = 0.07  # escala lineal de la marca (5-10% del canvas)
CALIBRATED  = 150
UNCALIB     = 350
MASK_THRESH = 8    # umbral de brillo para detectar la marca vs. fondo negro
RANDOM_SEED = 42
# ──────────────────────────────────────────────────────────────────────────────

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Cargar imágenes fuente
print("Cargando imágenes fuente...")
src_np = {}
for name, path in SRC.items():
    src_np[name] = np.array(Image.open(path).convert("RGB"))

H, W = src_np["cyan"].shape[:2]
print(f"Tamaño del canvas: {W}x{H}")

# Máscara de la marca negra = unión de los 3 canales de color
def bright_mask(arr, thresh=MASK_THRESH):
    return (arr[:,:,0] > thresh) | (arr[:,:,1] > thresh) | (arr[:,:,2] > thresh)

black_mask = (bright_mask(src_np["cyan"])
            | bright_mask(src_np["magenta"])
            | bright_mask(src_np["yellow"]))


Cargando imágenes fuente...


FileNotFoundError: [Errno 2] No such file or directory: '.\\registro_cyan_4k.png'

In [ ]:
# Sprites RGBA: marca de color sobre fondo transparente
def make_sprite(arr, mask=None):
    if mask is None:
        mask = bright_mask(arr)
    rgba = np.zeros((H, W, 4), dtype=np.uint8)
    rgba[mask, 0:3] = arr[mask]
    rgba[mask, 3]   = 255
    return rgba

# Sprite negro: RGB=0 (negro), alpha=255 donde está la marca
black_sprite = np.zeros((H, W, 4), dtype=np.uint8)
black_sprite[black_mask, 3] = 255

color_sprites = {name: make_sprite(src_np[name]) for name in src_np}

# ── Reducir la marca a SPRITE_SCALE del canvas ───────────────────────────────
SPR_W = max(1, round(W * SPRITE_SCALE))
SPR_H = max(1, round(H * SPRITE_SCALE))

def resize_sprite(sprite, sw, sh):
    """Escala RGB y máscara con NEAREST: bordes duros, sin sangrado de color."""
    rgb   = Image.fromarray(sprite[:, :, 0:3]).resize((sw, sh), Image.NEAREST)
    alpha = Image.fromarray(sprite[:, :, 3]).resize((sw, sh), Image.NEAREST)
    rgb, alpha = np.array(rgb), np.array(alpha)
    small = np.zeros((sh, sw, 4), dtype=np.uint8)
    m = alpha > 127
    small[m, 0:3] = rgb[m]
    small[m, 3]   = 255
    return small

black_sprite  = resize_sprite(black_sprite, SPR_W, SPR_H)
color_sprites = {name: resize_sprite(sp, SPR_W, SPR_H) for name, sp in color_sprites.items()}

In [ ]:
# ── Centro de cada marca (coordenadas dentro del sprite) ─────────────────────
def sprite_center(sprite):
    """Centro de la marca visible (bbox del alpha > 0) en coordenadas del sprite."""
    ys, xs = np.where(sprite[:, :, 3] > 0)
    if len(xs) == 0:
        return -1.0, -1.0
    return (xs.min() + xs.max()) / 2.0, (ys.min() + ys.max()) / 2.0

SPR_CENTER = {
    "black":   sprite_center(black_sprite),
    "cyan":    sprite_center(color_sprites["cyan"]),
    "magenta": sprite_center(color_sprites["magenta"]),
    "yellow":  sprite_center(color_sprites["yellow"]),
}

In [ ]:
# Compositing alpha sobre numpy array (overlay de cualquier tamaño, colocado en x, y)
def composite(base, overlay, x=0, y=0):
    sh, sw = overlay.shape[:2]
    BH, BW = base.shape[:2]
    src_x0 = max(-x, 0);     src_x1 = min(sw, BW - x)
    src_y0 = max(-y, 0);     src_y1 = min(sh, BH - y)
    dst_x0 = max(x, 0);      dst_x1 = min(x + sw, BW)
    dst_y0 = max(y, 0);      dst_y1 = min(y + sh, BH)
    if src_x0 >= src_x1 or src_y0 >= src_y1:
        return
    sr = overlay[src_y0:src_y1, src_x0:src_x1]
    dr = base[dst_y0:dst_y1, dst_x0:dst_x1]
    alpha = sr[:,:,3:4].astype(np.float32) / 255.0
    dr[:,:,0:3] = (sr[:,:,0:3] * alpha + dr[:,:,0:3] * (1 - alpha)).astype(np.uint8)

In [ ]:
# Funcion para elegir la posicion del registro negro aleatoriamente
def random_position():
    """Posición aleatoria tal que la marca + sus offsets (±OFFSET_MAX) quepan completos."""
    x = random.randint(OFFSET_MAX, W - SPR_W - OFFSET_MAX)
    y = random.randint(OFFSET_MAX, H - SPR_H - OFFSET_MAX)
    return x, y

# Funcion para desplazar un registro de color cian, magenta o azul con respeto con la posicion del registro negro
def random_offset():
    """Offset aleatorio con al menos un eje en [OFFSET_MIN, OFFSET_MAX]."""
    while True:
        dx = random.randint(-OFFSET_MAX, OFFSET_MAX)
        dy = random.randint(-OFFSET_MAX, OFFSET_MAX)

        # Solo retorna si las posiciones elegidas de x y y son mayores que el OFFSET_MIN
        if abs(dx) >= OFFSET_MIN or abs(dy) >= OFFSET_MIN:
            return dx, dy

In [ ]:
# Generar imágenes
rows = []
color_names = list(src_np.keys())  # ['cyan', 'magenta', 'yellow']

print(f"Generando {CALIBRATED} imágenes calibradas...")
for i in range(CALIBRATED):
    canvas = np.ones((H, W, 4), dtype=np.uint8) * 255  # fondo blanco
    bx, by = random_position()                         # registro negro en posición aleatoria
    composite(canvas, black_sprite, bx, by)
    img = Image.fromarray(canvas[:,:,:3], "RGB")
    filename = f"calibrated_{i:04d}.png"
    img.save(os.path.join(OUTPUT_DIR, filename))

    bcx = round(bx + SPR_CENTER["black"][0], 1)
    bcy = round(by + SPR_CENTER["black"][1], 1)
    
    rows.append({
        "filename":    filename,
        "calibrated":  1,
        "has_cyan":    0,
        "has_magenta": 0,
        "has_yellow":  0,
        "black_cx":    bcx,  "black_cy": bcy,
        "cyan_cx": -1.0, "cyan_cy": -1.0, "cyan_dx": -1, "cyan_dy": -1, "cyan_dist": -1.0,
        "magenta_cx": -1.0, "magenta_cy": -1.0, "magenta_dx": -1, "magenta_dy": -1, "magenta_dist": -1.0,
        "yellow_cx": -1.0, "yellow_cy": -1.0, "yellow_dx": -1, "yellow_dy": -1, "yellow_dist": -1.0,
    })

In [ ]:
print(f"Generando {UNCALIB} imágenes no calibradas...")
for i in range(UNCALIB):
    num_colors = random.randint(1, 3)
    chosen = random.sample(color_names, num_colors)

    canvas = np.ones((H, W, 4), dtype=np.uint8) * 255  # fondo blanco
    bx, by = random_position()                         # posición base de la marca negra
    

    offsets = {}
    
    # Canales de color desplazados respecto al negro (detrás)
    for ch in chosen:
        dx, dy = random_offset()
        offsets[ch] = (dx, dy)
        composite(canvas, color_sprites[ch], bx + dx, by + dy)

    # Registro negro encima
    composite(canvas, black_sprite, bx, by)

    img = Image.fromarray(canvas[:,:,:3], "RGB")
    filename = f"uncalibrated_{i:04d}.png"
    img.save(os.path.join(OUTPUT_DIR, filename))

    rows.append({
        "filename":    filename,
        "calibrated":  0,
        "has_cyan":    1 if "cyan"    in chosen else 0,
        "has_magenta": 1 if "magenta" in chosen else 0,
        "has_yellow":  1 if "yellow"  in chosen else 0,
    })

    row = {
        "filename":    filename,
        "calibrated":  0,
        "has_cyan":    1 if "cyan"    in chosen else 0,
        "has_magenta": 1 if "magenta" in chosen else 0,
        "has_yellow":  1 if "yellow"  in chosen else 0,
        "black_cx":    round(bx + SPR_CENTER["black"][0], 1),
        "black_cy":    round(by + SPR_CENTER["black"][1], 1),
    }
    for ch in color_names:
        cx0, cy0 = SPR_CENTER[ch]
        if ch in chosen:
            dx, dy = offsets[ch]
            row[f"{ch}_cx"] = round(bx + dx + cx0, 1)
            row[f"{ch}_cy"] = round(by + dy + cy0, 1)
            row[f"{ch}_dx"] = dx
            row[f"{ch}_dy"] = dy
            # aqui calculamos la distancia actual entre el color y el registro negro
            row[f"{ch}_dist"] = round(math.sqrt(dx * dx + dy * dy), 1)
        else:
            row[f"{ch}_cx"] = -1.0
            row[f"{ch}_cy"] = -1.0
            row[f"{ch}_dx"] = -1
            row[f"{ch}_dy"] = -1
            row[f"{ch}_dist"] = -1.0
    rows.append(row)

In [3]:
# Escribir CSV
print(f"Escribiendo CSV en {CSV_PATH}...")
fieldnames = [
    "filename", "calibrated",
    "has_cyan", "has_magenta", "has_yellow",
    "black_cx", "black_cy",
    "cyan_cx", "cyan_cy", "cyan_dx", "cyan_dy", "cyan_dist",
    "magenta_cx", "magenta_cy", "magenta_dx", "magenta_dy", "magenta_dist",
    "yellow_cx", "yellow_cy", "yellow_dx", "yellow_dy", "yellow_dist",
]
with open(CSV_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"\n✓ {len(rows)} imágenes guardadas en '{OUTPUT_DIR}/'")
print(f"✓ CSV guardado en '{CSV_PATH}'")

Escribiendo CSV en ./registro_dataset_v5.csv...


NameError: name 'rows' is not defined

In [9]:
pip install torch torchvision pandas scikit-learn matplotlib Pillow

Note: you may need to restart the kernel to use updated packages.


# Entrenando Modelo

In [10]:
"""
train_modelo.py
===============
Pipeline completo para entrenar un modelo que:
  1. Clasifica si una imagen está calibrada o no  (cabeza de clasificación)
  2. Predice el offset (dx, dy) de cada canal C/M/Y (cabeza de regresión)

Arquitectura: MobileNetV3-Small con dos cabezas de salida (multitask)
Requiere: torch, torchvision, pandas, scikit-learn, matplotlib, Pillow

Instalar dependencias:
  pip install torch torchvision pandas scikit-learn matplotlib Pillow

Uso:
  python train_modelo.py

El modelo entrenado se guarda en: ./modelo_registro.pth
Las métricas y curvas se guardan en: ./resultados/
"""

'\ntrain_modelo.py\n===============\nPipeline completo para entrenar un modelo que:\n  1. Clasifica si una imagen está calibrada o no  (cabeza de clasificación)\n  2. Predice el offset (dx, dy) de cada canal C/M/Y (cabeza de regresión)\n\nArquitectura: MobileNetV3-Small con dos cabezas de salida (multitask)\nRequiere: torch, torchvision, pandas, scikit-learn, matplotlib, Pillow\n\nInstalar dependencias:\n  pip install torch torchvision pandas scikit-learn matplotlib Pillow\n\nUso:\n  python train_modelo.py\n\nEl modelo entrenado se guarda en: ./modelo_registro.pth\nLas métricas y curvas se guardan en: ./resultados/\n'

In [11]:
"""
train_modelo.py
===============
Entrena un modelo multitask sobre el dataset de registros CMYK.
  - Clasificación binaria : calibrated  (0/1)
  - Clasificación binaria : has_cyan    (0/1)
  - Clasificación binaria : has_magenta (0/1)
  - Clasificación binaria : has_yellow  (0/1)

Uso:
  python train_modelo.py

Salida:
  ./modelo_registro.pth
  ./resultados/curvas.png
"""

'\ntrain_modelo.py\n===============\nEntrena un modelo multitask sobre el dataset de registros CMYK.\n  - Clasificación binaria : calibrated  (0/1)\n  - Clasificación binaria : has_cyan    (0/1)\n  - Clasificación binaria : has_magenta (0/1)\n  - Clasificación binaria : has_yellow  (0/1)\n\nUso:\n  python train_modelo.py\n\nSalida:\n  ./modelo_registro.pth\n  ./resultados/curvas.png\n'

In [12]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as T

In [13]:
warnings.filterwarnings('ignore')

# ── Configuración ─────────────────────────────────────────────────────────────
IMG_DIR     = './registro_dataset_1_8_26_v3'
CSV_PATH    = './registro_dataset_v3.csv'
MODEL_PATH  = './modelo_registro.pth'
RESULTS_DIR = './resultados'

IMG_SIZE     = 224
BATCH_SIZE   = 32
EPOCHS       = 30
LR           = 1e-4
WEIGHT_DECAY = 1e-4
SEED         = 42
# ──────────────────────────────────────────────────────────────────────────────

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(RESULTS_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')

LABEL_COLS = ['calibrated', 'has_cyan', 'has_magenta', 'has_yellow']


Dispositivo: cpu


In [14]:
# ── Dataset ───────────────────────────────────────────────────────────────────
class RegistroDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row['filename'])).convert('RGB')
        if self.transform: img = self.transform(img)
        labels = torch.tensor([float(row[c]) for c in LABEL_COLS], dtype=torch.float32)
        return img, labels

In [15]:
transform_train = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
transform_val = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [16]:
# ── Modelo ────────────────────────────────────────────────────────────────────
class RegistroModel(nn.Module):
    """
    MobileNetV3-Small → 4 salidas binarias independientes:
      [calibrated, has_cyan, has_magenta, has_yellow]
    """
    def __init__(self):
        super().__init__()
        bb = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
        self.features = bb.features
        self.avgpool  = bb.avgpool
        self.head = nn.Sequential(
            nn.Linear(576, 256),
            nn.Hardswish(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 4),   # 4 salidas binarias
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.head(x)     # logits [B, 4]

In [17]:
# ── Loss ──────────────────────────────────────────────────────────────────────
bce = nn.BCEWithLogitsLoss()

# ── Loops ─────────────────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct = 0, np.zeros(4)
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = bce(out, labels)
        loss.backward(); optimizer.step()
        total_loss += loss.item()
        preds = (torch.sigmoid(out) > 0.5).float()
        correct += (preds == labels).float().sum(0).cpu().numpy()
    n = len(loader)
    acc = correct / len(loader.dataset)
    return total_loss/n, acc   # acc shape [4]

In [18]:
@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    total_loss, correct = 0, np.zeros(4)
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out  = model(imgs)
        loss = bce(out, labels)
        total_loss += loss.item()
        preds = (torch.sigmoid(out) > 0.5).float()
        correct += (preds == labels).float().sum(0).cpu().numpy()
    n = len(loader)
    acc = correct / len(loader.dataset)
    return total_loss/n, acc

In [19]:
# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    df = pd.read_csv(CSV_PATH)
    print(f'Dataset: {len(df)} imágenes | calibradas: {df.calibrated.sum()} | '
          f'no calibradas: {(df.calibrated==0).sum()}')

    train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['calibrated'], random_state=SEED)
    val_df,   test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['calibrated'], random_state=SEED)
    print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

    kw = dict(num_workers=0, pin_memory=False)
    train_loader = DataLoader(RegistroDataset(train_df, IMG_DIR, transform_train), batch_size=BATCH_SIZE, shuffle=True,  **kw)
    val_loader   = DataLoader(RegistroDataset(val_df,   IMG_DIR, transform_val),   batch_size=BATCH_SIZE, shuffle=False, **kw)
    test_loader  = DataLoader(RegistroDataset(test_df,  IMG_DIR, transform_val),   batch_size=BATCH_SIZE, shuffle=False, **kw)

    model     = RegistroModel().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    history       = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
    best_val_loss = float('inf')

    header = f'{"Época":>5} | {"T-Loss":>8} | {"V-Loss":>8} | {"Cal":>6} | {"Cyan":>6} | {"Mag":>6} | {"Yel":>6}'
    print(f'\nEntrenando {EPOCHS} épocas en {DEVICE}...')
    print(header); print('─' * len(header))

    for epoch in range(1, EPOCHS + 1):
        tl, ta = train_epoch(model, train_loader, optimizer)
        vl, va = eval_epoch(model, val_loader)
        scheduler.step()
        history['train_loss'].append(tl); history['val_loss'].append(vl)
        history['train_acc'].append(ta);  history['val_acc'].append(va)
        saved = ''
        if vl < best_val_loss:
            best_val_loss = vl
            torch.save(model.state_dict(), MODEL_PATH)
            saved = ' ← guardado'
        print(f'{epoch:>5} | {tl:>8.4f} | {vl:>8.4f} | '
              f'{va[0]:>5.1%} | {va[1]:>5.1%} | {va[2]:>5.1%} | {va[3]:>5.1%}{saved}')

    # Test final
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    _, test_acc = eval_epoch(model, test_loader)
    print(f'\nTest accuracy:')
    for name, acc in zip(LABEL_COLS, test_acc):
        print(f'  {name:15}: {acc:.1%}')

    # Curvas
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history['train_loss'], label='Train'); axes[0].plot(history['val_loss'], label='Val')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Época')
    val_accs = np.array(history['val_acc'])   # shape [epochs, 4]
    for i, name in enumerate(LABEL_COLS):
        axes[1].plot(val_accs[:, i], label=name)
    axes[1].set_title('Val Accuracy por etiqueta'); axes[1].legend(); axes[1].set_xlabel('Época')
    axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'curvas.png'), dpi=120)
    plt.show()
    print(f'\nModelo guardado en  "{MODEL_PATH}"')
    print(f'Curvas guardadas en "{RESULTS_DIR}/curvas.png"')

if __name__ == '__main__':
    main()

Dataset: 12 imágenes | calibradas: 2 | no calibradas: 10


ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.